# Variáveis Estáticas

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B"
DATASET_PATH = "/content/drive/MyDrive/Datasets/LF-Amazon-1.3M/trn.json/sample.jsonl"
PROMPTING_TEMPLATE = "### Introduction:\n{instruction}\n### Question:\nDescribes the product: {title}\n"
RESPONSE_TEMPLATE = "### Response:\n{content}"
OUTPUT_DIR = "/content/drive/MyDrive/FIAP_TRANING_0210"
TRAINED_PATH = "/content/drive/MyDrive/FIAP_TRANING_0210/TRAINED"

# Instalação das Dependencias + Login no HuggingFace

In [ ]:
!pip install huggingface_hub peft transformers accelerate torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 35.5 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login(new_session=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Quantização

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                # quantize base model to 4 bits
    bnb_4bit_compute_dtype="float16", # use fp16 for computations
    bnb_4bit_use_double_quant=True,   # optional second quantization stage
    bnb_4bit_quant_type="nf4"         # NormalFloat4 (usually best for LLMs)
)

# Criação do Tokenizer e do Collator

In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Tokenizer converte texto ↔ tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

# Preparação do Dataset de Treino
## A idéia aqui é que a gente consegue montar o PROMPTING_TEMPLATE e o RESPONSE_TEMPLATE a qual quer momento então não me preocupei em fazer um dataset com eles para testar no final, somente com o prompt de treino completo para uso e descarte que é a pergunta + a resposta + eos_token

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# # Drop unused columns
# dataset = dataset.remove_columns(['uid', 'target_ind', 'target_rel'])
# # Remove empties
# dataset = dataset.filter(lambda row: row['title'].strip() != "" and row['content'].strip() != "")

# seen_titles = set()
# def keep_first(example):
#     title = example["title"].strip()
#     if title in seen_titles:
#         return False
#     seen_titles.add(title)
#     return True

# dataset = dataset.filter(keep_first, load_from_cache_file=False)

# Format prompt
def format_row(row):
    prompt_text = PROMPTING_TEMPLATE.format(instruction=row['instruction'], title=row['input']) \
    + RESPONSE_TEMPLATE.format(content=row['output']) \
    + tokenizer.eos_token
    return {"prompt": prompt_text}

dataset = dataset.map(format_row)

# Tokenize
def preprocess(batch):
  return tokenizer(batch["prompt"], truncation=True, padding=True, max_length=512)

tokenized_dataset = dataset.map(preprocess, batched=True, num_proc=2)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/30000 [00:00<?, ? examples/s]

# Testando o Prompting

In [ ]:
# Modelo (pode carregar em GPU se disponível)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             quantization_config=bnb_config,
                                             device_map="auto"
                                             )

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

...

Input:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling

Output from the untrained model:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling
### Answer:
Audel Managing Maintenance Planning and Scheduling is a comprehensive guide that provides valuable insights into the world of maintenance planning and scheduling. This book is designed to assist maintenance professionals in effectively managing their operations, ensuring optimal performance and efficiency.
The book begins by introducing the concept of maintenance planning and scheduling, highlighting its significance in the overall maintenance process. It explains the importance of effective planning and scheduling in reducing downtime, minimizing costs, and maximizing equipment uptime. The author emphasizes the need for a sy

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Input:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling

Output from the untrained model:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling
### Answer:
Audel Managing Maintenance Planning and Scheduling is a comprehensive guide that provides valuable insights and practical strategies for effectively managing maintenance planning and scheduling in various industries. This book is designed to assist maintenance professionals in optimizing their operations, reducing downtime, and maximizing equipment availability.
### Key Features:
1. **Comprehensive Coverage:** Audel Managing Maintenance Planning and Scheduling covers a wide range of topics related to maintenance planning and scheduling, including preventive maintenance, predictive maintenance, work order management, r

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Input:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling

Output from the untrained model:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling
### Answer:
Audel Managing Maintenance Planning and Scheduling is a book that provides comprehensive information on managing maintenance planning and scheduling. It covers various aspects of maintenance, including planning, scheduling, and execution. The book is designed to help maintenance managers, supervisors, and technicians to effectively manage their maintenance operations. It provides practical guidance on how to plan and schedule maintenance activities, optimize resources, and ensure that maintenance is carried out efficiently and effectively.
### Conclusion:
Audel Managing Maintenance Planning and Scheduling is a

In [ ]:
for i in range (0, 5):
  # Prompt de teste
  prompt = PROMPTING_TEMPLATE.format(instruction=dataset[i]['instruction'], title=dataset[i]['input'])

  # Tokenizar entrada
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  # Gerar texto
  outputs = model.generate(
      **inputs,
     max_new_tokens=250,
     temperature=0.3,   # controle de criatividade
     top_p=0.9,         # nucleus sampling
     do_sample=True
  )

  # Decodificar tokens → texto
  print(f'######### QUESTION {i}')
  print("Input:")
  print(prompt)
  print("Output from the untrained model:")
  print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


######### QUESTION 0
Input:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling

Output from the untrained model:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling
### Answer:
Audel Managing Maintenance Planning and Scheduling is a comprehensive guide that provides valuable insights and practical strategies for effective maintenance planning and scheduling. This book is designed to help professionals in the field of maintenance management to optimize their operations, reduce costs, and improve overall efficiency.
The book begins by introducing the concept of maintenance planning and scheduling, outlining its importance in modern industrial settings. It emphasizes the need for a systematic approach to maintenance, highlighting the benefits of proactive maintenance strategies ove

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


######### QUESTION 1
Input:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Reclaiming the Enlightenment: Toward a Politics of Radical Engagement

Output from the untrained model:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Reclaiming the Enlightenment: Toward a Politics of Radical Engagement
### Answer:
Reclaiming the Enlightenment: Toward a Politics of Radical Engagement is a book that delves into the profound significance of the Enlightenment era and its relevance to contemporary society. Written by renowned philosopher and activist, Reclaiming the Enlightenment: Toward a Politics of Radical Engagement offers a critical analysis of the Enlightenment's impact on politics, society, and culture. The book explores the historical context of the Enlightenment, its key figures, and the ideas that shaped this intellectual movement. Through a comprehensive examination of En

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


######### QUESTION 2
Input:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: R&#65533;sle Wall Attachment with Cap, Stainless Steel

Output from the untrained model:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: R&#65533;sle Wall Attachment with Cap, Stainless Steel
### Answer:
R&#65533;sle Wall Attachment with Cap, Stainless Steel
R&#65533;sle Wall Attachment with Cap, Stainless Steel
The R&#65533;sle Wall Attachment with Cap, Stainless Steel is a versatile and practical tool designed to enhance the functionality of your kitchen or workspace. This stainless steel attachment is specifically designed to be used with the R&#65533;sle Wall Mount, providing a secure and convenient way to store and access your favorite kitchen tools and utensils.
One of the key features of the R&#65533;sle Wall Attachment with Cap is its compatibility with the R&#65533;sle Wall Mount.

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


######### QUESTION 3
Input:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Q3973A HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)

Output from the untrained model:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Q3973A HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)
### Answer:
The HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner) is a high-quality printing solution designed to enhance your printing experience. This cartridge is specifically designed for the HP Color LaserJet 2840 Series printers and is known for its exceptional performance and reliability. With a yield of 2000 pages, this cartridge ensures that you can print a significant number of documents without the need for frequent replacements.
The HP Col

# Configurando LoRA para aplicar no treinamento

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=64,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM" # tipo de tarefa
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.print_trainable_parameters()

trainable params: 27,262,976 || all params: 8,057,524,224 || trainable%: 0.3384


# Treinando

In [ ]:
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd

# Split the dataset into training and evaluation sets
train_test_split = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]


training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,     # ← onde salvar
        save_total_limit=2,        # ← quantos checkpoints manter
        save_steps=100,            # ← salva a cada 50 steps
        per_device_train_batch_size=20,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=10,
        learning_rate=5e-4,
        fp16=False,
        bf16=True,
        logging_steps=100,
        optim="paged_adamw_8bit",
        report_to="none"
    )

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=data_collator
)


In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss


In [ ]:
model.save_pretrained(TRAINED_PATH)

# Carregando e Testando

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

trained_model = PeftModel.from_pretrained(model, TRAINED_PATH)

In [ ]:
for i in range (0, 5):
  # Prompt de teste
  prompt = PROMPTING_TEMPLATE.format(instruction=dataset[i]['instruction'], title=dataset[i]['input'])

  # Tokenizar entrada
  inputs = tokenizer(prompt, return_tensors="pt").to(trained_model.device)

  # Gerar texto
  outputs2 = trained_model.generate(
      **inputs,
     max_new_tokens=250,
     temperature=0.3,   # controle de criatividade
     top_p=0.9,         # nucleus sampling
     do_sample=True
  )

  print(f'\n######### QUESTION {i}')
  print("Input:")
  print(prompt)
  print("Output from the trained model:")
  print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



######### QUESTION 0
Input:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling

Output from the trained model:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Audel Managing Maintenance Planning and Scheduling
### Response:
A guide to developing, implementing, and administering a maintenance plan that is both cost-effective and efficient.
This book covers the basics of planning and scheduling, and explores the latest technologies and software to help you manage a maintenance plan that will keep a facility running smoothly and profitably.
Topics include:Developing a mission statement and objectivesEstimating work quantities and costsPrioritizing workAssigning resourcesEstimating time and moneyAllocating capital for preventive maintenancePurchasing spare partsScheduling work with othersInsuring adequate supervisionCoordinatin

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



######### QUESTION 1
Input:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Reclaiming the Enlightenment: Toward a Politics of Radical Engagement

Output from the trained model:
### Introduction:
Create a clear and attractive product description.
### Question:
Describes the product: Reclaiming the Enlightenment: Toward a Politics of Radical Engagement
### Response:
"An important contribution to the ongoing debate about the legacy of Enlightenment political theory and the meaning of 'critical theory' today.
Habermas's engagement with Rousseau, Kant, and the early Frankfurt School is always insightful, and his critique of the 'communicative distortions' of modern consumer society is incisive.
"&#x2014;Richard Bernstein, author of The New Constellation: The Ethics of Postmodernism"Habermas's engagement with Rousseau, Kant, and the early Frankfurt School is always insightful, and his critique of the 'communicative distortions' of m

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



######### QUESTION 2
Input:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: R&#65533;sle Wall Attachment with Cap, Stainless Steel

Output from the trained model:
### Introduction:
Explain the main benefits and features of the product.
### Question:
Describes the product: R&#65533;sle Wall Attachment with Cap, Stainless Steel
### Response:
This wall attachment is designed for use with R sle's 1-1/2-inch diameter stainless steel spiral pipe. It consists of a cap with a spiral pipe receptacle and a separate wall attachment with a 1/4-inch NPT female thread receptacle.
The cap is used when the spiral pipe is to be terminated at a wall and the spiral pipe is to be continued through the NPT receptacle. The wall attachment is used when the spiral pipe is to be terminated at a wall and the NPT receptacle is to be used for another purpose, such as for pressure gauge connection.
Spiral pipe must be cut to length and slipped over the

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



######### QUESTION 3
Input:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Q3973A HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)

Output from the trained model:
### Introduction:
Describe the product concisely and in a human-like manner.
### Question:
Describes the product: Q3973A HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)
### Response:
Q3973A HP Color LJ 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)MagNeta 2000 Yield - (2 Year Warranty) - (2# Q3973A). Q3973A HP Color LaserJet 2840 Series Smart Printer Cartridge Magenta (2000 Yield) - (Genuine OEM toner)MagNeta 2000 Yield - (2 Year Warranty) - (2# Q3973A).
Compatible with: Color LJ 2840 Series Smart Printer - (C9398A) - (Genuine OEM toner) - (2 Q3968X, 1 Q3970X, 1 Q3971X, 1 Q3972X, 1 Q3973A, 1 Q3974A) - (2# Q3968X, 1 Q3970X, 1 Q3971X,

In [ ]:
for i in range (0, 5):
  print('\n')
  print(dataset[i]['output'])



A good plan is good for businessBreakdown maintenance still accounts for much of the time maintenance workers put in. Too often, the result is lost revenue, excessive downtime, and poor-quality repairs.
This convenient, practical guide shows you how to develop a comprehensive planning and scheduling effort to ensure all resources are available when they are needed.
You&#8217;ll discover how to gather supportive data and build plans that will help you control maintenance costs and equipment downtime.
Make informed decisions about the most effective way to perform maintenanceEstablish solid shutdown schedulesSet reasonable goals based on your budgetUnderstand a range of estimating and scheduling methods Structure a work order system that supports your planAllocate money, material, and labor resources for maximum productivityUse multi-skill training to its best advantageFormulate methods to identify the right work to be performed during a shutdown


Stephen Bronner has written a much-ne